# Notebook 02 - Descriptive analytics dataset_v05 (TK-3)

Bahan untuk **subbagian 4.1 Gambaran data** dan metode deskriptif di 3.2 (rubrik kriteria 2, *descriptive analytics*).
Notebook ini tidak memakai label anotasi, jadi bisa dijalankan kapan saja.

Isi:
1. Karakteristik dataset
2. Pola temporal bulanan dan peristiwa pemicu
3. Engagement dan konsentrasi
4. Tagar dan istilah pada caption video
5. Ringkasan angka (satu sumber angka untuk laporan)

**Unit analisis.** Kecuali disebut lain, hitungan memakai record `eligible == True`.
"Komentar" = `item_type == comment` (tanggapan langsung ke video), "balasan" = `item_type == reply` (tanggapan ke komentar).
Waktu dalam UTC, sama dengan `report.md`.

**Etika.** Notebook ini tidak menampilkan teks komentar maupun ID apa pun. Yang keluar hanya angka agregat, tagar, dan istilah caption video.
File di `outputs/` aman di-commit; data di `data/processed/` tetap tidak masuk repo.

**Cara menjalankan.** Buka dari root repo atau dari folder `notebooks/`. Bila dataset disimpan di tempat lain (mis. hasil unduhan dari Drive kelompok),
atur dulu variabel lingkungan `DATASET_PATH` ke file `dataset_v05.parquet`; file `dataset_v05.metadata.json` harus ada di folder yang sama.

In [1]:
from pathlib import Path
from collections import Counter
import json
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter

ROOT = Path.cwd()
if not (ROOT / "data").exists():          # dijalankan dari notebooks/
    ROOT = ROOT.parent
DATA = Path(os.environ.get("DATASET_PATH", ROOT / "data" / "processed" / "dataset_v05.parquet"))
FIG = ROOT / "outputs" / "figures"
TAB = ROOT / "outputs" / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

AMBANG_BULAN_KECIL = 100   # bulan dengan komentar+balasan di bawah ini tidak ditafsirkan
AMBANG_VIDEO_JARANG = 1    # bulan dengan video diunggah <= ini diarsir di Gambar 1

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

In [2]:
# Format angka Indonesia: 18.636 dan 38,2%
def angka(x, des=0):
    s = f"{x:,.{des}f}"
    return s.replace(",", "_").replace(".", ",").replace("_", ".")

def persen(x, des=1):
    return angka(100 * x, des) + "%"

NAMA_BULAN = ["Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]

# Kategori tagar. "Generik platform" = tagar algoritma/tren yang tidak menyebut topik.
POLA_FIGUR = re.compile(r"teddy|tedsky|seskab|sekretaris|letkol|mayor|^tiw")
POLA_GENERIK = re.compile(r"^fy|^foryou|^viral|^trending|^xyzbca|^capcut|^masukberanda|^tiktok$|^beranda")

def kategori_tagar(t):
    if POLA_FIGUR.search(t):
        return "Merujuk figur"
    if POLA_GENERIK.search(t):
        return "Generik platform"
    return "Lainnya"

def rapikan_tagar(t):
    # fyp, fypシ, fypage, fyppppp... dihitung sebagai satu tagar
    return "fyp (semua varian)" if t.startswith("fyp") else t

# Gaya gambar: palet kategorikal tervalidasi (slot 1-3), teks abu-abu, grid tipis
BIRU, ORANYE, AQUA = "#257de0", "#f47230", "#00b894"
TEKS, TEKS2, ABU, ARSIR = "#191919", "#666666", "#8a8a8a", "#f1f1f1"
plt.rcParams.update({
    "font.size": 8.5, "axes.titlesize": 9, "axes.labelsize": 8.5,
    "axes.edgecolor": TEKS2, "axes.labelcolor": TEKS, "text.color": TEKS,
    "xtick.color": TEKS2, "ytick.color": TEKS2, "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e3e3e3", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "legend.frameon": False, "legend.fontsize": 8,
    "savefig.dpi": 300, "savefig.bbox": "tight", "figure.dpi": 110,
})
fmt_ribuan = FuncFormatter(lambda v, _: angka(v))
fmt_persen = FuncFormatter(lambda v, _: angka(100 * v) + "%")

def simpan_gambar(fig, nama):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{nama}.{ext}")
    print("Disimpan:", f"outputs/figures/{nama}.png dan .pdf")

def simpan_tabel(tbl, nama, index=True):
    tbl.to_csv(TAB / f"{nama}.csv", index=index, encoding="utf-8")
    print("Disimpan:", f"outputs/tables/{nama}.csv")

## 0. Memuat data

Jumlah record dicocokkan dengan `metadata.json`. Bila tidak sama, berarti file yang dimuat bukan dataset_v05 yang dipakai laporan.

In [3]:
def ke_list(v):
    # parquet menyimpan tagar sebagai array; CSV menyimpannya sebagai teks
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return []
    if isinstance(v, (list, tuple, np.ndarray)):
        return [str(t) for t in v]
    s = str(v).strip()
    if s.startswith("["):
        try:
            return [str(t) for t in json.loads(s)]
        except ValueError:
            pass
    return [t for t in re.split(r"[\s,;'\"\[\]]+", s) if t]

df = pd.read_parquet(DATA) if DATA.suffix == ".parquet" else pd.read_csv(DATA, low_memory=False)
for c in ("item_id", "parent_id", "conversation_id", "author_hash"):
    df[c] = df[c].astype("string")
df["created_at"] = pd.to_datetime(df["created_at"], utc=True, errors="coerce")
df["hashtags"] = df["hashtags"].apply(ke_list)
if df["eligible"].dtype != bool:
    df["eligible"] = df["eligible"].astype(str).str.lower().eq("true")
for c in ("like_count", "reply_count", "share_count", "view_count"):
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("float")

meta = json.loads(DATA.with_name(DATA.name.split(".")[0] + ".metadata.json").read_text(encoding="utf-8"))
PERIODE = meta["keywords_snapshot"]["periode"]
PERISTIWA = meta["keywords_snapshot"]["peristiwa_pemicu"]
KUERI_TAGAR = set(meta["keywords_snapshot"]["hashtag"])

langkah = meta["steps"]
assert len(df) == sum(v["dataset"] for v in langkah.values()), "Jumlah record berbeda dengan metadata"
assert df["eligible"].sum() == sum(v["eligible"] for v in langkah.values()), "Jumlah eligible berbeda dengan metadata"

el = df[df["eligible"]].copy()
video_semua = df[df["item_type"] == "post"].set_index("item_id")   # semua video, untuk tanggal unggah & jumlah komentar asli
video = el[el["item_type"] == "post"].copy()                       # video eligible
kom = el[el["item_type"].isin(["comment", "reply"])].copy()        # komentar + balasan eligible
kom["bulan"] = kom["created_at"].dt.tz_localize(None).dt.to_period("M")

R = {"dataset": meta["dataset"], "periode": PERIODE}   # ringkasan angka, ditulis di bagian 5
print(f"{meta['dataset']}: {angka(len(df))} record, {angka(len(el))} eligible. Cocok dengan metadata.")

dataset_v05: 21.650 record, 18.783 eligible. Cocok dengan metadata.


## 1. Karakteristik dataset

Beberapa hal yang perlu dibaca bersama tabel:
- Sebagian kecil video tidak eligible (caption kosong atau caption berbahasa lain), tetapi komentarnya tetap eligible bila komentarnya lolos kriteria.
  Karena itu jumlah video yang punya komentar eligible bisa lebih besar daripada jumlah video eligible.
- **Cakupan pengambilan.** `reply_count` pada video adalah jumlah komentar yang dilaporkan TikTok saat pengambilan.
  Scraper hanya mengambil halaman komentar teratas (urut suka), jadi rasio terkumpul/dilaporkan menunjukkan seberapa kecil potongan percakapan yang kita amati.

In [4]:
tipe = pd.DataFrame({
    "dataset": df["item_type"].value_counts(),
    "eligible": el["item_type"].value_counts(),
}).reindex(["post", "comment", "reply"]).fillna(0).astype(int)
tipe["porsi_eligible"] = (tipe["eligible"] / tipe["eligible"].sum()).round(4)
tipe.index = ["video (post)", "komentar (comment)", "balasan (reply)"]
tipe.loc["total"] = [tipe["dataset"].sum(), tipe["eligible"].sum(), 1.0]
tipe[["dataset", "eligible"]] = tipe[["dataset", "eligible"]].astype(int)
simpan_tabel(tipe.rename_axis("jenis_item"), "desk_01a_jenis_item")
tipe

Disimpan: outputs/tables/desk_01a_jenis_item.csv


,dataset,eligible,porsi_eligible
video (post),153,147,0.0078
komentar (comment),10702,9350,0.4978
balasan (reply),10795,9286,0.4944
total,21650,18783,1.0000


In [5]:
# Cakupan pengambilan: komentar yang terkumpul (semua, sebelum filter) dibanding jumlah komentar yang dilaporkan video
terkumpul = df[df["item_type"] == "comment"].groupby("conversation_id").size()
cakupan = pd.concat({"terkumpul": terkumpul, "dilaporkan": video_semua["reply_count"]}, axis=1).dropna()
cakupan = cakupan[cakupan["dilaporkan"] > 0]
cakupan["rasio"] = (cakupan["terkumpul"] / cakupan["dilaporkan"]).clip(upper=1)

per_video = kom.groupby("conversation_id").size()
per_akun = kom.groupby("author_hash").size()
kata = kom["text_clean"].fillna("").str.split().str.len()

bendera = [c for c in df.columns if c.startswith("flag_")]
R.update({
    "total_record": int(len(df)),
    "record_eligible": int(len(el)),
    "video_eligible": int(len(video)),
    "video_dengan_komentar_eligible": int(per_video.size),
    "komentar_eligible": int((kom["item_type"] == "comment").sum()),
    "balasan_eligible": int((kom["item_type"] == "reply").sum()),
    "komentar_balasan_eligible": int(len(kom)),
    "akun_unik_komentar_balasan": int(per_akun.size),
    "porsi_akun_satu_komentar": round(float((per_akun == 1).mean()), 4),
    "median_kata_komentar": float(kata[kom["item_type"] == "comment"].median()),
    "median_kata_balasan": float(kata[kom["item_type"] == "reply"].median()),
    "median_komentar_balasan_per_video": float(per_video.median()),
    "maks_komentar_balasan_per_video": int(per_video.max()),
    "komentar_terkumpul": int(cakupan["terkumpul"].sum()),
    "komentar_dilaporkan_platform": int(cakupan["dilaporkan"].sum()),
    "cakupan_agregat": round(float(cakupan["terkumpul"].sum() / cakupan["dilaporkan"].sum()), 4),
    "cakupan_median_per_video": round(float(cakupan["rasio"].median()), 4),
    "flag_eksklusi": {c: int(df[c].sum()) for c in bendera if df[c].sum() > 0},
})

karakteristik = pd.DataFrame([
    ("Record dalam dataset", angka(R["total_record"]), "sebelum filter eligible"),
    ("Record eligible", angka(R["record_eligible"]), persen(R["record_eligible"] / R["total_record"]) + " dari dataset"),
    ("Video eligible", angka(R["video_eligible"]), ""),
    ("Video dengan komentar eligible", angka(R["video_dengan_komentar_eligible"]), "termasuk video yang caption-nya tidak eligible"),
    ("Komentar eligible", angka(R["komentar_eligible"]), "unit analisis sentimen dan topik (RQ2)"),
    ("Balasan eligible", angka(R["balasan_eligible"]), "unit analisis RQ3"),
    ("Akun unik (hash) penulis komentar/balasan", angka(R["akun_unik_komentar_balasan"]),
     persen(R["porsi_akun_satu_komentar"]) + " hanya menulis satu kali"),
    ("Median kata per komentar / balasan", f"{angka(R['median_kata_komentar'])} / {angka(R['median_kata_balasan'])}", "dari text_clean"),
    ("Median komentar+balasan per video", angka(R["median_komentar_balasan_per_video"], 0 if R["median_komentar_balasan_per_video"] % 1 == 0 else 1), f"maks. {angka(R['maks_komentar_balasan_per_video'])}"),
    ("Komentar terkumpul / dilaporkan platform", f"{angka(R['komentar_terkumpul'])} / {angka(R['komentar_dilaporkan_platform'])}",
     f"cakupan {persen(R['cakupan_agregat'])}; median per video {persen(R['cakupan_median_per_video'])}"),
], columns=["indikator", "nilai", "catatan"])
simpan_tabel(karakteristik, "desk_01b_karakteristik", index=False)
karakteristik

Disimpan: outputs/tables/desk_01b_karakteristik.csv


,indikator,nilai,catatan
0,Record dalam dataset,21.650,sebelum filter eligible
1,Record eligible,18.783,"86,8% dari dataset"
2,Video eligible,147,
3,Video dengan komentar eligible,148,termasuk video yang caption-nya tidak eligible
4,Komentar eligible,9.350,unit analisis sentimen dan topik (RQ2)
5,Balasan eligible,9.286,unit analisis RQ3
6,Akun unik (hash) penulis komentar/balasan,16.136,"90,7% hanya menulis satu kali"
7,Median kata per komentar / balasan,6 / 5,dari text_clean
8,Median komentar+balasan per video,"84,5",maks. 850
9,Komentar terkumpul / dilaporkan platform,10.702 / 222.285,"cakupan 4,8%; median per video 24,6%"


## 2. Pola temporal bulanan dan peristiwa pemicu

Tiga hal dihitung:
- **Jumlah per bulan** untuk video yang diunggah, komentar, dan balasan, beserta porsinya. Bulan dengan komentar+balasan < `AMBANG_BULAN_KECIL` ditandai dan tidak ditafsirkan.
- **Jeda komentar** dari waktu unggah video. Bila sebagian besar komentar ditulis beberapa hari setelah video diunggah, pola bulanan komentar pada dasarnya mengikuti kapan video populer diunggah.
- **Peristiwa pemicu** dari `keywords.yaml`: jumlah pada jendela setelah peristiwa dibanding jendela yang sama panjang sebelum peristiwa.

In [6]:
bulan_all = pd.period_range(PERIODE["mulai"], PERIODE["selesai"], freq="M")
vid_bulan = video["created_at"].dt.tz_localize(None).dt.to_period("M").value_counts()
kom_bulan = kom.groupby(["bulan", "item_type"]).size().unstack(fill_value=0).reindex(bulan_all, fill_value=0)

bulanan = pd.DataFrame(index=bulan_all)
bulanan["video_diunggah"] = vid_bulan.reindex(bulan_all, fill_value=0).astype(int)
bulanan["komentar"] = kom_bulan.get("comment", 0)
bulanan["balasan"] = kom_bulan.get("reply", 0)
bulanan["total"] = bulanan["komentar"] + bulanan["balasan"]
bulanan["porsi_total"] = (bulanan["total"] / bulanan["total"].sum()).round(4)
bulanan["n_kecil"] = bulanan["total"] < AMBANG_BULAN_KECIL
bulanan["video_jarang"] = bulanan["video_diunggah"] <= AMBANG_VIDEO_JARANG
bulanan.index.name = "bulan"
assert bulanan["total"].sum() == len(kom), "Ada komentar di luar periode"

tahun = bulanan.groupby(bulanan.index.year)[["video_diunggah", "komentar", "balasan", "total"]].sum()
tahun["porsi_total"] = (tahun["total"] / tahun["total"].sum()).round(4)
paruh1 = bulanan["total"].iloc[:12].sum()   # 12 bulan pertama periode

R.update({
    "bulan_n_kecil": [str(b) for b in bulanan.index[bulanan["n_kecil"]]],
    "porsi_komentar_balasan_2026": round(float(tahun.loc[2026, "porsi_total"]), 4) if 2026 in tahun.index else None,
    "porsi_komentar_balasan_12_bulan_pertama": round(float(paruh1 / bulanan["total"].sum()), 4),
    "video_diunggah_per_tahun": {int(k): int(v) for k, v in tahun["video_diunggah"].items()},
})
simpan_tabel(bulanan.assign(bulan=bulanan.index.astype(str)).set_index("bulan"), "desk_02a_bulanan")
print(f"Porsi komentar+balasan pada 12 bulan pertama periode: {persen(R['porsi_komentar_balasan_12_bulan_pertama'])}")
print(f"Bulan dengan n < {AMBANG_BULAN_KECIL}: {', '.join(R['bulan_n_kecil'])}")
tahun

Disimpan: outputs/tables/desk_02a_bulanan.csv
Porsi komentar+balasan pada 12 bulan pertama periode: 4,3%
Bulan dengan n < 100: 2024-11, 2024-12, 2025-01, 2025-03, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09


,video_diunggah,komentar,balasan,total,porsi_total
bulan,,,,,
2024,1,118,160,278,0.0149
2025,13,951,271,1222,0.0656
2026,133,8281,8855,17136,0.9195


In [7]:
# Jeda antara unggah video dan komentar/balasan
kom["video_at"] = kom["conversation_id"].map(video_semua["created_at"])
kom["jeda_hari"] = (kom["created_at"] - kom["video_at"]).dt.total_seconds() / 86400
jeda = kom["jeda_hari"].dropna()

R["jeda_hari"] = {
    "median": round(float(jeda.median()), 2),
    "p75": round(float(jeda.quantile(.75)), 2),
    "p90": round(float(jeda.quantile(.90)), 2),
    "porsi_dalam_1_hari": round(float((jeda <= 1).mean()), 4),
    "porsi_dalam_7_hari": round(float((jeda <= 7).mean()), 4),
    "porsi_dalam_30_hari": round(float((jeda <= 30).mean()), 4),
    "negatif": int((jeda < 0).sum()),   # seharusnya 0
}
tabel_jeda = pd.DataFrame({
    "komentar": kom.loc[kom["item_type"] == "comment", "jeda_hari"].describe(percentiles=[.25, .5, .75, .9]),
    "balasan": kom.loc[kom["item_type"] == "reply", "jeda_hari"].describe(percentiles=[.25, .5, .75, .9]),
}).round(2)
simpan_tabel(tabel_jeda.rename_axis("statistik_jeda_hari"), "desk_02b_jeda_komentar")
print(f"Median jeda {angka(R['jeda_hari']['median'], 1)} hari; "
      f"{persen(R['jeda_hari']['porsi_dalam_7_hari'])} ditulis dalam 7 hari setelah video diunggah.")
tabel_jeda

Disimpan: outputs/tables/desk_02b_jeda_komentar.csv
Median jeda 1,5 hari; 73,8% ditulis dalam 7 hari setelah video diunggah.


,komentar,balasan
count,9350.00,9286.00
mean,16.59,10.09
std,40.27,35.78
min,0.00,0.01
25%,0.42,0.45
50%,1.78,1.30
75%,14.54,4.95
90%,51.03,23.71
max,590.46,593.72


In [8]:
# Bulan puncak: berapa banyak video yang menyumbang, dan tagar (non-generik) dari video penyumbang terbesar.
# Gunanya untuk menelusuri pemicu puncak yang tidak tercantum di keywords.yaml; konfirmasi peristiwanya tetap lewat sumber berita.
baris = []
for b in bulanan.nlargest(3, "total").index:
    sub = kom[kom["bulan"] == b]
    per_vid = sub.groupby("conversation_id").size().sort_values(ascending=False)
    tagar = Counter(
        rapikan_tagar(t)
        for i in per_vid.index[:3] if i in video_semua.index
        for t in video_semua.at[i, "hashtags"]
        if kategori_tagar(t) != "Generik platform"
    )
    baris.append({
        "bulan": str(b),
        "komentar_balasan": int(len(sub)),
        "video_diunggah_bulan_itu": int(bulanan.at[b, "video_diunggah"]),
        "video_yang_dikomentari": int(per_vid.size),
        "porsi_video_teratas": round(float(per_vid.iloc[0] / len(sub)), 4),
        "porsi_3_video_teratas": round(float(per_vid.iloc[:3].sum() / len(sub)), 4),
        "tagar_3_video_teratas": ", ".join(f"#{t}" for t, _ in tagar.most_common(6)),
    })
puncak = pd.DataFrame(baris)
simpan_tabel(puncak, "desk_02c_bulan_puncak", index=False)
puncak

Disimpan: outputs/tables/desk_02c_bulan_puncak.csv


,bulan,komentar_balasan,video_diunggah_bulan_itu,video_yang_dikomentari,porsi_video_teratas,porsi_3_video_teratas,tagar_3_video_teratas
0,2026-06,4070,30,69,0.1157,0.3152,"#prabowo, #creatorsearchinsights, #teddyindraw..."
1,2026-09,3393,32,97,0.1491,0.3602,"#metropolitanid, #seskabteddy, #prabowo, #arah..."
2,2026-07,2103,11,58,0.2877,0.5925,"#teddyindrawijaya, #inews, #teddy, #akmil, #ma..."


In [9]:
NAMA_PERISTIWA = {
    "pengumuman_kabinet": "Pengumuman\nkabinet",
    "kenaikan_pangkat_letkol": "Kenaikan\npangkat",
    "viral_social_experiment": "Video social\nexperiment viral",
}
baris = []
for ev in PERISTIWA:
    t0 = pd.Timestamp(ev["tanggal"], tz="UTC")
    d = pd.Timedelta(days=ev["jendela_hari"])
    sesudah = kom["created_at"].between(t0, t0 + d, inclusive="left")
    sebelum = kom["created_at"].between(t0 - d, t0, inclusive="left")
    v_sesudah = video["created_at"].between(t0, t0 + d, inclusive="left")
    v_sebelum = video["created_at"].between(t0 - d, t0, inclusive="left")
    baris.append({
        "peristiwa": NAMA_PERISTIWA.get(ev["nama"], ev["nama"]).replace("\n", " ").replace("_", " "),
        "tanggal": ev["tanggal"],
        "jendela_hari": ev["jendela_hari"],
        "video_sebelum": int(v_sebelum.sum()),
        "video_sesudah": int(v_sesudah.sum()),
        "komentar_balasan_sebelum": int(sebelum.sum()),
        "komentar_balasan_sesudah": int(sesudah.sum()),
        "rasio_sesudah_sebelum": round(sesudah.sum() / sebelum.sum(), 2) if sebelum.sum() else np.nan,
    })
peristiwa = pd.DataFrame(baris)
simpan_tabel(peristiwa, "desk_02d_peristiwa_pemicu", index=False)
peristiwa

Disimpan: outputs/tables/desk_02d_peristiwa_pemicu.csv


,peristiwa,tanggal,jendela_hari,video_sebelum,video_sesudah,komentar_balasan_sebelum,komentar_balasan_sesudah,rasio_sesudah_sebelum
0,Pengumuman kabinet,2024-10-20,14,1,0,69,139,2.01
1,Kenaikan pangkat,2025-02-25,14,1,0,159,8,0.05
2,Video social experiment viral,2026-05-13,14,5,6,934,481,0.51


### Gambar 1 — volume bulanan

Panel atas: komentar dan balasan eligible per bulan (menurut waktu komentar ditulis). Panel bawah: video eligible yang diunggah per bulan.
Bulan yang diarsir hanya punya paling banyak satu video, sehingga jumlah komentarnya mencerminkan satu video, bukan percakapan umum.
Dua panel dipakai, bukan dua sumbu-y, karena skalanya berbeda jauh.

In [10]:
x0 = bulanan.index.to_timestamp()
xm = x0 + pd.Timedelta(days=15)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 4.2), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1.15], "hspace": 0.1})
for ax in (ax1, ax2):
    for xi, j in zip(x0, bulanan["video_jarang"]):
        if j:
            ax.axvspan(xi, xi + pd.offsets.MonthBegin(1), color=ARSIR, lw=0, zorder=0)
    ax.grid(axis="x", visible=False)

ax1.bar(xm, bulanan["komentar"], width=24, color=BIRU, edgecolor="white", linewidth=0.8, label="Komentar")
ax1.bar(xm, bulanan["balasan"], width=24, bottom=bulanan["komentar"], color=ORANYE, edgecolor="white", linewidth=0.8, label="Balasan")
ymax = bulanan["total"].max() * 1.16
ax1.set_ylim(0, ymax)
ax1.yaxis.set_major_formatter(fmt_ribuan)
ax1.set_ylabel("Komentar dan balasan")

# Label peristiwa: digeser turun bila berdekatan (< 200 hari) dengan peristiwa sebelumnya,
# dan ditulis di kiri garis bila peristiwa berada di sepertiga akhir periode.
tengah = x0[0] + (x0[-1] - x0[0]) * 2 / 3
t_sebelum, tingkat = None, 0
for ev in PERISTIWA:
    t = pd.Timestamp(ev["tanggal"])
    for ax in (ax1, ax2):
        ax.axvline(t, color=TEKS2, ls=(0, (3, 2)), lw=0.8, zorder=1)
    tingkat = tingkat + 1 if t_sebelum is not None and (t - t_sebelum).days < 200 else 0
    kiri = t > tengah
    ax1.text(t + pd.Timedelta(days=-6 if kiri else 6), ymax * (0.98 - 0.12 * tingkat),
             NAMA_PERISTIWA.get(ev["nama"], ev["nama"]), fontsize=7, color=TEKS2, linespacing=1.1,
             va="top", ha="right" if kiri else "left")
    t_sebelum = t

ax2.bar(xm, bulanan["video_diunggah"], width=24, color=ABU, edgecolor="white", linewidth=0.8)
ax2.set_ylabel("Video\ndiunggah")
ax2.set_ylim(0, bulanan["video_diunggah"].max() * 1.2)

tick = [t for t in x0 if t.month in (1, 4, 7, 10)]
ax2.set_xticks(tick)
ax2.set_xticklabels([f"{NAMA_BULAN[t.month - 1]}\n{t.year}" for t in tick])
ax2.set_xlim(x0[0], x0[-1] + pd.offsets.MonthBegin(1))

ax1.legend(handles=[
    Patch(color=BIRU, label="Komentar"), Patch(color=ORANYE, label="Balasan"),
    Patch(facecolor=ARSIR, edgecolor="#d0d0d0", label=f"Bulan dengan ≤{AMBANG_VIDEO_JARANG} video"),
    Line2D([0], [0], color=TEKS2, ls=(0, (3, 2)), lw=0.8, label="Peristiwa pemicu"),
], loc="lower left", bbox_to_anchor=(0, 1.0), ncol=4, handlelength=1.4, columnspacing=1.2)
simpan_gambar(fig, "desk_gambar1_volume_bulanan")
plt.show()

Disimpan: outputs/figures/desk_gambar1_volume_bulanan.png dan .pdf


## 3. Engagement dan konsentrasi

- **Engagement video** (tayang, suka, komentar, bagikan) adalah angka kumulatif saat pengambilan (Oktober 2026).
  Video lama punya waktu lebih panjang untuk mengumpulkan engagement, jadi angka ini tidak dibandingkan antarperiode.
- **Engagement komentar** hanya punya jumlah suka; jumlah balasan hanya ada pada komentar.
- **Konsentrasi**: seberapa besar porsi yang dikuasai unit teratas, untuk tiga hubungan: video → komentar+balasan, komentar → suka, dan akun → komentar+balasan.
  Gini 0 berarti merata, 1 berarti terkumpul pada satu unit.

In [11]:
def ringkas(s):
    s = s.dropna()
    return pd.Series({"n": len(s), "median": s.median(), "p75": s.quantile(.75), "p90": s.quantile(.90), "maks": s.max()})

eng_video = pd.DataFrame({
    "tayang": ringkas(video["view_count"]),
    "suka": ringkas(video["like_count"]),
    "komentar_dilaporkan": ringkas(video["reply_count"]),
    "bagikan": ringkas(video["share_count"]),
}).T.round(1)
rasio_suka_tayang = (video["like_count"] / video["view_count"]).replace([np.inf, -np.inf], np.nan).dropna()
R["engagement_video"] = {k: {kk: float(vv) for kk, vv in v.items()} for k, v in eng_video.to_dict("index").items()}
R["median_rasio_suka_per_tayang_video"] = round(float(rasio_suka_tayang.median()), 4)
simpan_tabel(eng_video.rename_axis("metrik"), "desk_03a_engagement_video")
print(f"Median rasio suka/tayang per video: {persen(R['median_rasio_suka_per_tayang_video'])}")
eng_video

Disimpan: outputs/tables/desk_03a_engagement_video.csv
Median rasio suka/tayang per video: 4,3%


,n,median,p75,p90,maks
tayang,147.0,635100.0,2200000.0,5080000.0,24000000.0
suka,147.0,18700.0,80100.0,185340.0,844800.0
komentar_dilaporkan,147.0,316.0,1784.5,5137.2,11300.0
bagikan,147.0,1086.0,5711.0,21100.0,198100.0


In [12]:
def porsi_teratas(nilai, p=None, k=None):
    v = np.sort(np.asarray(nilai, dtype=float))[::-1]
    if k is None:
        k = max(1, int(np.ceil(p * len(v))))
    return float(v[:k].sum() / v.sum())

def gini(nilai):
    v = np.sort(np.asarray(nilai, dtype=float))
    n = len(v)
    return float(2 * np.sum(np.arange(1, n + 1) * v) / (n * v.sum()) - (n + 1) / n)

suka_kom = kom.loc[kom["item_type"] == "comment", "like_count"].dropna()
suka_bal = kom.loc[kom["item_type"] == "reply", "like_count"].dropna()
eng_kom = pd.DataFrame({
    "komentar": ringkas(suka_kom),
    "balasan": ringkas(suka_bal),
}).T
eng_kom["porsi_tanpa_suka"] = [(suka_kom == 0).mean(), (suka_bal == 0).mean()]
eng_kom["porsi_suka_1pct_teratas"] = [porsi_teratas(suka_kom, .01), porsi_teratas(suka_bal, .01)]
eng_kom = eng_kom.round(4)

balasan_dilaporkan = kom.loc[kom["item_type"] == "comment", "reply_count"].dropna()
R["engagement_komentar"] = {k: {kk: float(vv) for kk, vv in v.items()} for k, v in eng_kom.to_dict("index").items()}
R["porsi_komentar_berbalasan"] = round(float((balasan_dilaporkan > 0).mean()), 4)
simpan_tabel(eng_kom.rename_axis("jenis"), "desk_03b_suka_komentar")
print(f"Komentar eligible yang punya minimal satu balasan (menurut platform): {persen(R['porsi_komentar_berbalasan'])}")
eng_kom

Disimpan: outputs/tables/desk_03b_suka_komentar.csv
Komentar eligible yang punya minimal satu balasan (menurut platform): 35,6%


,n,median,p75,p90,maks,porsi_tanpa_suka,porsi_suka_1pct_teratas
komentar,9350.0,2.0,13.0,143.0,227651.0,0.3003,0.7509
balasan,9286.0,1.0,4.0,22.0,20244.0,0.4856,0.6940


In [13]:
unit = {
    "Video → komentar+balasan": per_video,
    "Komentar → suka": suka_kom,
    "Akun → komentar+balasan": per_akun,
}
konsentrasi = pd.DataFrame([{
    "hubungan": nama,
    "n_unit": int(len(v)),
    "total": float(np.sum(v)),
    "porsi_1pct_teratas": round(porsi_teratas(v, .01), 4),
    "porsi_10pct_teratas": round(porsi_teratas(v, .10), 4),
    "porsi_5_unit_teratas": round(porsi_teratas(v, k=5), 4),
    "gini": round(gini(v), 3),
} for nama, v in unit.items()])
R["konsentrasi"] = konsentrasi.set_index("hubungan").drop(columns="total").to_dict("index")
simpan_tabel(konsentrasi, "desk_03c_konsentrasi", index=False)
konsentrasi

Disimpan: outputs/tables/desk_03c_konsentrasi.csv


,hubungan,n_unit,total,porsi_1pct_teratas,porsi_10pct_teratas,porsi_5_unit_teratas,gini
0,Video → komentar+balasan,148,18636.0,0.0787,0.3832,0.1631,0.524
1,Komentar → suka,9350,6977376.0,0.7509,0.9888,0.1305,0.979
2,Akun → komentar+balasan,16136,18636.0,0.0536,0.2208,0.0065,0.126


### Gambar 2 — kurva konsentrasi

Sumbu-x: porsi unit teratas (diurutkan dari yang terbesar). Sumbu-y: porsi kumulatif dari total.
Garis putus-putus adalah kondisi merata. Semakin cepat kurva naik, semakin terkonsentrasi.

In [14]:
def kurva(nilai):
    v = np.sort(np.asarray(nilai, dtype=float))[::-1]
    return np.r_[0, np.arange(1, len(v) + 1) / len(v)], np.r_[0, np.cumsum(v) / v.sum()]

fig, ax = plt.subplots(figsize=(3.4, 3.0))
ax.plot([0, 1], [0, 1], color=ABU, ls=(0, (3, 2)), lw=0.8)
ax.text(0.74, 0.66, "Merata", color=TEKS2, fontsize=7, rotation=38, ha="center", va="center")
for (nama, v), warna in zip(unit.items(), (BIRU, ORANYE, AQUA)):
    xs, ys = kurva(v)
    ax.plot(xs, ys, color=warna, lw=2, label=nama, solid_capstyle="round")
    y10 = porsi_teratas(v, .10)
    ax.plot([0.10], [y10], "o", ms=5, color=warna, mec="white", mew=1.2, zorder=3)
    ax.text(0.125, y10, persen(y10, 0), fontsize=7, color=TEKS, va="center")
ax.axvline(0.10, color=ABU, lw=0.6, ls=":")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.xaxis.set_major_formatter(fmt_persen)
ax.yaxis.set_major_formatter(fmt_persen)
ax.set_xlabel("Porsi unit teratas")
ax.set_ylabel("Porsi kumulatif dari total")
ax.legend(loc="lower right", handlelength=1.4)
simpan_gambar(fig, "desk_gambar2_konsentrasi")
plt.show()

Disimpan: outputs/figures/desk_gambar2_konsentrasi.png dan .pdf


## 4. Tagar dan istilah pada caption video

Dihitung per video: satu video dihitung sekali untuk setiap tagar atau istilah, berapa pun kemunculannya.
Varian `#fyp` (`#fypシ`, `#fypage`, dst.) digabung.

**Perlu dibaca hati-hati.** Empat tagar dan empat kata kunci dipakai sebagai kueri pengumpulan data, sehingga frekuensinya
sebagian merupakan hasil desain pengambilan, bukan semata pilihan pembuat video. Tagar kueri diberi tanda `*` di Gambar 3.

In [15]:
tagar_video = video["hashtags"].apply(lambda hs: sorted({rapikan_tagar(t) for t in hs}))
hitung_tagar = Counter(t for hs in tagar_video for t in hs)
tagar = pd.DataFrame(hitung_tagar.most_common(), columns=["tagar", "jumlah_video"])
tagar["porsi_video"] = (tagar["jumlah_video"] / len(video)).round(4)
tagar["kategori"] = tagar["tagar"].map(lambda t: "Generik platform" if t.startswith("fyp") else kategori_tagar(t))
tagar["kueri_pengumpulan"] = tagar["tagar"].isin(KUERI_TAGAR)

tanpa_tagar = int((tagar_video.str.len() == 0).sum())
kat_video = tagar_video.apply(lambda hs: {("Generik platform" if t.startswith("fyp") else kategori_tagar(t)) for t in hs})
R["tagar_caption"] = {
    "video_tanpa_tagar": tanpa_tagar,
    "tagar_unik": int(len(tagar)),
    "porsi_video_bertagar_figur": round(float(kat_video.apply(lambda s: "Merujuk figur" in s).mean()), 4),
    "porsi_video_bertagar_generik": round(float(kat_video.apply(lambda s: "Generik platform" in s).mean()), 4),
    "top10": tagar.head(10)[["tagar", "jumlah_video"]].values.tolist(),
}
simpan_tabel(tagar.head(30), "desk_04a_tagar_caption", index=False)
print(f"{angka(len(video))} video, {tanpa_tagar} tanpa tagar, {angka(len(tagar))} tagar unik.")
tagar.head(20)

Disimpan: outputs/tables/desk_04a_tagar_caption.csv
147 video, 6 tanpa tagar, 189 tagar unik.


,tagar,jumlah_video,porsi_video,kategori,kueri_pengumpulan
0,teddyindrawijaya,68,0.4626,Merujuk figur,True
1,fyp (semua varian),64,0.4354,Generik platform,False
2,mayorteddy,55,0.3741,Merujuk figur,True
3,letkolteddy,44,0.2993,Merujuk figur,True
4,seskabteddyindrawijaya,35,0.2381,Merujuk figur,False
5,prabowo,28,0.1905,Lainnya,False
6,teddy,16,0.1088,Merujuk figur,False
7,seskabteddy,14,0.0952,Merujuk figur,True
8,viral,13,0.0884,Generik platform,False
9,capcut,12,0.0816,Generik platform,False


In [16]:
# Cara caption (teks + tagar) menyebut figur: pangkat militer, jabatan sipil, atau keduanya.
# Ketiga sebutan juga ada di kata kunci pengumpulan, jadi angkanya tidak mencerminkan populasi video.
teks_video = (video["text_clean"].fillna("") + " " + video["hashtags"].apply(" ".join)).str.lower()
pangkat = teks_video.str.contains(r"\bmayor\b|\bletkol\b|mayorteddy|letkolteddy", regex=True)
jabatan = teks_video.str.contains(r"seskab|sekretaris\s*kabinet|sekretariskabinet", regex=True)
sebutan = pd.DataFrame({
    "jumlah_video": [int((pangkat & ~jabatan).sum()), int((jabatan & ~pangkat).sum()), int((pangkat & jabatan).sum()), int((~pangkat & ~jabatan).sum())],
}, index=["pangkat militer saja", "jabatan sipil saja", "keduanya", "tidak keduanya (nama saja/lainnya)"])
sebutan["porsi_video"] = (sebutan["jumlah_video"] / len(video)).round(4)
R["sebutan_figur_caption"] = sebutan["jumlah_video"].to_dict()
simpan_tabel(sebutan.rename_axis("sebutan"), "desk_04b_sebutan_figur")
sebutan

Disimpan: outputs/tables/desk_04b_sebutan_figur.csv


,jumlah_video,porsi_video
pangkat militer saja,54,0.3673
jabatan sipil saja,40,0.2721
keduanya,40,0.2721
tidak keduanya (nama saja/lainnya),13,0.0884


In [17]:
STOPWORD = set('''
yang dan di ke dari ini itu dengan untuk pada adalah dalam tidak akan juga ada sudah saya kami kita aku kamu dia ia
mereka ya yg aja nya lagi jadi bisa kalau kalo atau karena tapi tetapi saat oleh agar supaya seperti sangat lebih
masih belum harus hanya pun para sang si dgn utk krn tdk gak ga nggak enggak ngga sih deh kok dong loh lho kan nih tuh
tersebut mau apa tak sebuah jangan depan langsung ketika bagaimana sambil begitu saja kini hari satu dua baru
banyak buat bikin sama semua video lihat terus
the of and to in for on is a an with this that you my your be it at by from as are was
'''.split())
KUERI_KATA = {w for k in meta["keywords_snapshot"]["kata_kunci"] for w in k.lower().split()}

def token_caption(s):
    s = re.sub(r"#\S+|\[URL\]|\[USER\]", " ", s or "")
    return {t for t in re.findall(r"[a-z]+", s.lower()) if len(t) >= 3 and t not in STOPWORD}

hitung_kata = Counter(t for s in video["text_clean"] for t in token_caption(s))
istilah = pd.DataFrame(hitung_kata.most_common(25), columns=["istilah", "jumlah_video"])
istilah["porsi_video"] = (istilah["jumlah_video"] / len(video)).round(4)
istilah["kata_kunci_pengumpulan"] = istilah["istilah"].isin(KUERI_KATA)
simpan_tabel(istilah, "desk_04c_istilah_caption", index=False)
istilah

Disimpan: outputs/tables/desk_04c_istilah_caption.csv


,istilah,jumlah_video,porsi_video,kata_kunci_pengumpulan
0,teddy,50,0.3401,True
1,seskab,32,0.2177,True
2,kabinet,20,0.1361,False
3,indra,20,0.1361,True
4,wijaya,20,0.1361,True
5,sekretaris,19,0.1293,False
6,presiden,19,0.1293,False
7,pak,17,0.1156,False
8,prabowo,15,0.1020,False
9,subianto,10,0.0680,False


### Gambar 3 — tagar caption terbanyak

In [18]:
top = tagar.head(15).iloc[::-1]
warna_kat = {"Merujuk figur": BIRU, "Lainnya": ORANYE, "Generik platform": ABU}
label = [f"#{t}{' *' if q else ''}" for t, q in zip(top["tagar"], top["kueri_pengumpulan"])]
fig, ax = plt.subplots(figsize=(3.4, 3.6))
ax.barh(range(len(top)), top["jumlah_video"], height=0.62, color=top["kategori"].map(warna_kat), edgecolor="white", linewidth=0.8)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(label, fontsize=7.5, color=TEKS)
ax.tick_params(axis="y", length=0)
ax.grid(axis="y", visible=False)
ax.set_xlabel(f"Jumlah video (dari {angka(len(video))})")
ax.legend(handles=[Patch(color=w, label=k) for k, w in warna_kat.items() if k in set(top["kategori"])],
          loc="lower right", handlelength=1.2)
ax.text(1.0, -0.22, "* tagar yang dipakai sebagai kueri pengumpulan", transform=ax.transAxes,
        ha="right", fontsize=6.5, color=TEKS2)
simpan_gambar(fig, "desk_gambar3_tagar_caption")
plt.show()

Disimpan: outputs/figures/desk_gambar3_tagar_caption.png dan .pdf


## 5. Ringkasan angka

`outputs/tables/desk_ringkasan.json` adalah **satu sumber angka** untuk bagian Dataset, Hasil 4.1, Abstrak, dan README.
Bila dataset berubah, jalankan ulang notebook ini dan ambil angka dari file tersebut, jangan dari ingatan atau draf lama.

In [19]:
def ke_json(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if np.isnan(o) else round(float(o), 4)
    if isinstance(o, float):
        return None if np.isnan(o) else round(o, 4)
    if isinstance(o, dict):
        return {str(k): ke_json(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [ke_json(v) for v in o]
    return o

R["dibuat_dari"] = str(DATA.name)
(TAB / "desk_ringkasan.json").write_text(json.dumps(ke_json(R), ensure_ascii=False, indent=2), encoding="utf-8")
print("Disimpan: outputs/tables/desk_ringkasan.json")
print(json.dumps(ke_json({k: R[k] for k in ("record_eligible", "komentar_eligible", "balasan_eligible",
                                            "video_dengan_komentar_eligible", "cakupan_agregat",
                                            "porsi_komentar_balasan_12_bulan_pertama")}), indent=2))

Disimpan: outputs/tables/desk_ringkasan.json
{
  "record_eligible": 18783,
  "komentar_eligible": 9350,
  "balasan_eligible": 9286,
  "video_dengan_komentar_eligible": 148,
  "cakupan_agregat": 0.0481,
  "porsi_komentar_balasan_12_bulan_pertama": 0.0431
}


## Catatan untuk penulisan

- **Bias temporal.** Laporkan jumlah per bulan bersama porsinya. Bulan bertanda `n_kecil` dan bulan yang diarsir di Gambar 1 tidak ditafsirkan sebagai perubahan percakapan.
- **Komentar mengikuti video.** Karena sebagian besar komentar ditulis dalam beberapa hari setelah video diunggah, naik-turunnya volume bulanan lebih mencerminkan kapan video populer diunggah daripada perubahan minat publik. Sampaikan ini sebagai sifat data, bukan temuan tentang figur.
- **Puncak di luar daftar peristiwa.** Bila bulan puncak tidak berdekatan dengan peristiwa di `keywords.yaml`, jangan menebak pemicunya. Telusuri lewat sumber berita lalu sitasi; tagar di `desk_02c_bulan_puncak.csv` hanya petunjuk awal.
- **Cakupan.** Komentar yang terkumpul hanya sebagian kecil dari komentar yang dilaporkan platform dan condong ke komentar paling disukai. Ini batasan untuk RQ1–RQ3, bukan hanya deskriptif.
- **Tagar kueri.** Frekuensi tagar dan sebutan figur sebagian ditentukan kata kunci pengumpulan.
- **Netral.** Angka engagement menggambarkan perhatian terhadap konten, bukan dukungan atau penilaian terhadap figur.